# Semantic video retrieval: complete reproducible walkthrough

Install `python -m pip install -e ".[dev]"` and optionally `jupyterlab matplotlib`. This walkthrough runs the implemented pinned pretrained CLIP indexing and query pipeline on licensed footage. Index construction does not fine-tune CLIP. The DiDeMo learned-adapter study is separate ongoing work; this notebook makes no unexecuted training claim. No media or weights are embedded.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib
from datetime import datetime, timezone
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from its repository.'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))
RUN_REAL_PIPELINE = False  # Set True to download/process real data and create fresh artifacts.
RUN = ROOT / 'data' / 'interim' / ('notebook-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
print('Repository:', ROOT, '\nFresh run:', RUN)


## Inspect encoder identity and recorded evaluation
The eight-query movie-held-out result is historical and small. Reading its JSON is not a fresh test evaluation; do not tune against those observed labels.

In [ ]:
from src.config import MODEL_ID, MODEL_REVISION
print({'model': MODEL_ID, 'immutable_revision': MODEL_REVISION})
historical = json.loads((ROOT / 'models/heldout-evaluation.json').read_text())
print({'queries': historical['queries'], 'recorded_metrics': historical['metrics'], 'selection_sha256': historical['selection_sha256']})

## Fetch pinned licensed real footage
Enable RUN_REAL_PIPELINE to download and encode. The model download is approximately 605 MB. Work in a unique run directory so production summaries and old evidence are preserved. The source function verifies the video checksum and writes attribution.

In [ ]:
from contextlib import contextmanager
@contextmanager
def working_directory(path):
    old = Path.cwd()
    os.chdir(path)
    try: yield
    finally: os.chdir(old)

if RUN_REAL_PIPELINE:
    RUN.mkdir(parents=True, exist_ok=False)
    from src.data.make_dataset import fetch
    with working_directory(RUN):
        provenance = fetch(RUN / 'data/raw/big-buck-bunny.mp4')
    print(provenance)
else:
    print('Media download and encoder inference disabled; set RUN_REAL_PIPELINE=True.')

## Sample and encode frames; persist the index
This calls the production implementation: actual pretrained image embeddings, normalization, deduplication, and integrity metadata. No task-specific training is performed.

In [ ]:
if RUN_REAL_PIPELINE:
    from src.models.train_model import build
    with working_directory(RUN):
        summary = build([RUN / 'data/raw/big-buck-bunny.mp4'], RUN / 'index', interval=2., device='cpu')
    print(json.dumps(summary, indent=2))

## Reload and search with natural language
The loader validates vector integrity and query encoder identity. Similarity is not a probability. Filenames and timestamps provide traceability, not learned labels.

In [ ]:
if RUN_REAL_PIPELINE:
    from src.models.index import VectorIndex
    from src.models.predict_model import search
    index = VectorIndex.load(RUN / 'index')
    matches = search(index, 'An animated bird singing on a tree branch', k=5)
    print(json.dumps(matches, indent=2))
    (RUN / 'query-results.json').write_text(json.dumps(matches, indent=2, allow_nan=False))

## Visualize historical held-out aggregate metrics
The chart uses previously executed results. It does not run the held-out scorer or imply a large-scale benchmark.

In [ ]:
metrics = historical['metrics']
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(list(metrics), list(metrics.values()))
    ax.set_ylim(0, 1)
    ax.set_title('Recorded eight-query movie-held-out study')
    plt.tight_layout()
except ImportError:
    print(metrics)

## Real-world benchmark and limits
See `docs/didemo-protocol.md` and its access amendment for the real human-annotated DiDeMo study. Availability filtering is frozen before model outcomes; missing public videos create selection bias. This walkthrough does not release or score its test labels. CLIP can miss actions, temporal relations and small objects. Run `python -m pytest -q` for unit mechanics. Fresh runs remain local under ignored data/.